In [ ]:
# final_analysis.py
# 包含残差分析、t-SNE、Input x Grad原子贡献、残基聚合、配体vs蛋白对比、电性属性相关性分析、多目标散点 + Pearson、残基类别归因统计（支持批量样本分析）

import os
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from sklearn.manifold import TSNE
from torch_geometric.loader import DataLoader
from GNN_model import PocketGNN1
from sklearn.utils import shuffle
from collections import defaultdict

residue_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
                'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','LIG']

# ======= Residual Analysis =======
def residual_analysis(y_true, y_pred, save_dir, param_names=['kcat', 'Km']):
    os.makedirs(save_dir, exist_ok=True)
    residuals = y_true - y_pred
    for i, name in enumerate(param_names):
        res = residuals[:, i]
        plt.figure()
        plt.hist(res, bins=30, edgecolor='black')
        plt.title(f'{name} Residual Histogram')
        plt.xlabel('Residual')
        plt.ylabel('Frequency')
        plt.savefig(os.path.join(save_dir, f'{name}_residual_hist.png'))
        plt.close()

        plt.figure()
        stats.probplot(res, dist="norm", plot=plt)
        plt.title(f'{name} Residual Q-Q Plot')
        plt.savefig(os.path.join(save_dir, f'{name}_residual_qq.png'))
        plt.close()

# ======= Input Gradient Attribution =======
def compute_input_gradient(model, data, target_index=0):
    model.eval()
    data.x.requires_grad_(True)
    output = model(data)
    score = output[:, target_index].sum()
    grad = torch.autograd.grad(score, data.x, retain_graph=True)[0]
    attribution = (data.x * grad).sum(dim=1)
    return attribution.detach().cpu().numpy()

# ======= Node Attribution Bar =======
def visualize_node_attribution(attribution, save_path='atom_importance_bar.png', topk=20):
    topk_idx = np.argsort(np.abs(attribution))[-topk:][::-1]
    topk_vals = attribution[topk_idx]
    plt.figure(figsize=(10, 6))
    sns.barplot(x=np.arange(topk), y=topk_vals)
    plt.title("Top-k Atom Contribution (Input x Gradient)")
    plt.xlabel("Atom Index")
    plt.ylabel("Importance Score")
    plt.savefig(save_path)
    plt.close()

# ======= Residue Class Attribution =======
def residue_class_importance(attribution, data, save_path='residue_class_bar.png'):
    if not hasattr(data, 'x') or not hasattr(data, 'pdb_id'):
        return
    residue_onehot = data.x[:, 10:31].detach().cpu().numpy() 
    contrib = attribution.reshape(-1, 1) * residue_onehot
    contrib_sum = contrib.sum(axis=0)
    contrib_dict = {res: val for res, val in zip(residue_list, contrib_sum)}
    sorted_items = sorted(contrib_dict.items(), key=lambda x: abs(x[1]), reverse=True)
    labels, values = zip(*sorted_items)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(labels), y=list(values))
    plt.xticks(rotation=45)
    plt.title("Residue-level Aggregated Attribution")
    plt.ylabel("Total Contribution")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ======= Batch Residue Class Attribution (Mean) =======
def average_residue_contribution(dataset, model, device, target_index=0, save_path='residue_class_avg.png'):
    model.eval()
    total_contrib = defaultdict(float)
    counts = defaultdict(int)
    for data in dataset:
        data = data.to(device)
        try:
            attrib = compute_input_gradient(model, data, target_index)
            residue_onehot = data.x[:, 10:31].detach().cpu().numpy()
            contrib = attrib.reshape(-1, 1) * residue_onehot
            for i, res in enumerate(residue_list):
                val = contrib[:, i].sum()
                total_contrib[res] += val
                counts[res] += 1
        except:
            continue
    contrib_avg = {res: total_contrib[res] / counts[res] if counts[res] else 0 for res in residue_list}
    sorted_items = sorted(contrib_avg.items(), key=lambda x: abs(x[1]), reverse=True)
    labels, values = zip(*sorted_items)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(labels), y=list(values))
    plt.xticks(rotation=45)
    plt.title("Mean Residue Contribution (All Samples)")
    plt.ylabel("Avg Attribution Score")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ======= kcat vs Km Scatter with Pearson =======
def plot_kcat_km_scatter(y_true, y_pred, save_path='scatter_kcat_km.png'):
    kcat_pred = y_pred[:, 0]
    km_pred = y_pred[:, 1]
    kcat_true = y_true[:, 0]
    km_true = y_true[:, 1]
    plt.figure(figsize=(8, 6))
    plt.scatter(kcat_pred, km_pred, label='Predicted', alpha=0.6)
    plt.scatter(kcat_true, km_true, label='True', alpha=0.6)
    plt.xlabel('kcat')
    plt.ylabel('Km')
    plt.legend()
    plt.title('kcat vs Km Prediction Distribution')
    plt.savefig(save_path)
    plt.close()

    r = np.corrcoef(kcat_pred, km_pred)[0,1]
    with open(save_path.replace('.png', '_pearson.txt'), 'w') as f:
        f.write(f"Pearson correlation between kcat and Km predictions: {r:.4f}\n")

# ======= t-SNE Visualization (with corrected colorbar) =======
def tsne_visualization(model, data_loader, device, save_path='tsne_kcat.png', param_index=0, max_points=1000):
    model.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            emb = model.get_graph_embedding(batch)
            embeddings.append(emb.cpu())
            labels.append(batch.y.view(-1, 2)[:, param_index].cpu())
    X = torch.cat(embeddings).numpy()
    Y = torch.cat(labels).numpy()
    X, Y = shuffle(X, Y, random_state=42)
    X, Y = X[:max_points], Y[:max_points]

    tsne = TSNE(n_components=2, perplexity=30)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(10, 8))
    sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=Y, cmap='coolwarm', s=60)
    plt.colorbar(sc)
    plt.title("Graph Embedding t-SNE (colored by target)")
    plt.savefig(save_path)
    plt.close()

# ======= 主入口 =======
if __name__ == '__main__':
    DATASET_PATH = '../data/processed/dataset_NAN_nopqr.pt'
    MODEL_PATH = '../outputs/nopqr/best_model.pt'
    SAVE_DIR = '../analysis_results/nopqr_more'
    DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    BATCH_SIZE = 8

    dataset = torch.load(DATASET_PATH)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = PocketGNN1(node_input_dim=dataset[0].x.shape[1], edge_input_dim=dataset[0].edge_attr.shape[1])
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)

    # ==== 残差分析 & 多目标散点图 ====
    all_y_true, all_y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            y = batch.y.view(-1, 2).cpu()
            pred = model(batch).cpu()
            all_y_true.append(y)
            all_y_pred.append(pred)

    all_y_true = torch.cat(all_y_true).numpy()
    all_y_pred = torch.cat(all_y_pred).numpy()
    residual_analysis(all_y_true, all_y_pred, SAVE_DIR)
    plot_kcat_km_scatter(all_y_true, all_y_pred, os.path.join(SAVE_DIR, 'scatter_kcat_km.png'))

    # ==== t-SNE for kcat and Km ====
    tsne_visualization(model, loader, DEVICE, save_path=os.path.join(SAVE_DIR, 'tsne_kcat.png'), param_index=0)
    tsne_visualization(model, loader, DEVICE, save_path=os.path.join(SAVE_DIR, 'tsne_km.png'), param_index=1)

    # ==== 单个样本归因分析 ====
    data = dataset[0].to(DEVICE)
    attribution = compute_input_gradient(model, data, target_index=0)
    visualize_node_attribution(attribution, save_path=os.path.join(SAVE_DIR, 'node_importance_bar.png'))
    residue_class_importance(attribution, data, save_path=os.path.join(SAVE_DIR, 'residue_class_bar.png'))

    # ==== 平均残基贡献分析 ====
    average_residue_contribution(dataset, model, DEVICE, target_index=0, save_path=os.path.join(SAVE_DIR, 'residue_class_avg.png'))


/tmp/ipykernel_3467651/895217867.py:162: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset = torch.load(DATASET_PATH)
/tmp/ipykernel_3467651/895217867.py:166: FutureWarn

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [3]:
# final_analysis.py
# 包含残差分析、t-SNE、Input x Grad原子贡献、残基聚合、配体vs蛋白对比、电性属性相关性分析、多目标散点 + Pearson、残基类别归因统计、样本归因热图、嵌入距离与误差分析（支持批量样本分析）

import os
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from torch_geometric.loader import DataLoader
from GNN_model import PocketGNN1
from sklearn.utils import shuffle
from collections import defaultdict

residue_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
                'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','LIG']

# ======= Residual Analysis =======
def residual_analysis(y_true, y_pred, save_dir, param_names=['kcat', 'Km']):
    os.makedirs(save_dir, exist_ok=True)
    residuals = y_true - y_pred
    for i, name in enumerate(param_names):
        res = residuals[:, i]
        plt.figure()
        plt.hist(res, bins=30, edgecolor='black')
        plt.title(f'{name} Residual Histogram')
        plt.xlabel('Residual')
        plt.ylabel('Frequency')
        plt.savefig(os.path.join(save_dir, f'{name}_residual_hist.png'))
        plt.close()

        plt.figure()
        stats.probplot(res, dist="norm", plot=plt)
        plt.title(f'{name} Residual Q-Q Plot')
        plt.savefig(os.path.join(save_dir, f'{name}_residual_qq.png'))
        plt.close()

# ======= Input Gradient Attribution =======
def compute_input_gradient(model, data, target_index=0):
    model.eval()
    data.x.requires_grad_(True)
    output = model(data)
    score = output[:, target_index].sum()
    grad = torch.autograd.grad(score, data.x, retain_graph=True)[0]
    attribution = (data.x * grad).sum(dim=1)
    return attribution.detach().cpu().numpy()

# ======= Node Attribution Bar =======
def visualize_node_attribution(attribution, save_path='atom_importance_bar.png', topk=20):
    topk_idx = np.argsort(np.abs(attribution))[-topk:][::-1]
    topk_vals = attribution[topk_idx]
    plt.figure(figsize=(10, 6))
    sns.barplot(x=np.arange(topk), y=topk_vals)
    plt.title("Top-k Atom Contribution (Input x Gradient)")
    plt.xlabel("Atom Index")
    plt.ylabel("Importance Score")
    plt.savefig(save_path)
    plt.close()

# ======= Residue Class Attribution =======
def residue_class_importance(attribution, data, save_path='residue_class_bar.png'):
    if not hasattr(data, 'x') or not hasattr(data, 'pdb_id'):
        return
    residue_onehot = data.x[:, 10:31].detach().cpu().numpy() 
    contrib = attribution.reshape(-1, 1) * residue_onehot
    contrib_sum = contrib.sum(axis=0)
    contrib_dict = {res: val for res, val in zip(residue_list, contrib_sum)}
    sorted_items = sorted(contrib_dict.items(), key=lambda x: abs(x[1]), reverse=True)
    labels, values = zip(*sorted_items)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(labels), y=list(values))
    plt.xticks(rotation=45)
    plt.title("Residue-level Aggregated Attribution")
    plt.ylabel("Total Contribution")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ======= Residue × Sample Attribution Matrix Heatmap =======
def attribution_matrix_heatmap(dataset, model, device, save_path='residue_heatmap.png', target_index=0):
    model.eval()
    heatmap = []
    for data in dataset:
        try:
            data = data.to(device)
            attrib = compute_input_gradient(model, data, target_index)
            residue_onehot = data.x[:, 10:31].detach().cpu().numpy()
            contrib = attrib.reshape(-1, 1) * residue_onehot
            contrib_sum = contrib.sum(axis=0)
            heatmap.append(contrib_sum)
        except:
            continue
    heatmap = np.stack(heatmap)
    plt.figure(figsize=(12, 6))
    sns.heatmap(heatmap, xticklabels=residue_list, cmap='coolwarm', center=0)
    plt.xlabel("Residue Type")
    plt.ylabel("Sample")
    plt.title("Attribution Matrix (Residue × Sample)")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ======= Embedding Distance vs Error Scatter =======
def embedding_distance_vs_error(dataset, model, device, save_path='embedding_error_scatter.png'):
    model.eval()
    embeddings, errors = [], []
    with torch.no_grad():
        for data in dataset:
            data = data.to(device)
            emb = model.get_graph_embedding(data).cpu().numpy().flatten()
            pred = model(data).cpu().numpy().flatten()
            true = data.y.cpu().numpy().flatten()
            err = np.abs(pred - true).mean()
            embeddings.append(emb)
            errors.append(err)
    dists = pairwise_distances(embeddings)
    err_mean = np.mean(errors)
    dist_flat, err_flat = dists.flatten(), np.repeat(errors, len(errors))
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=dist_flat, y=err_flat, alpha=0.4)
    plt.xlabel("Pairwise Embedding Distance")
    plt.ylabel("Prediction Error")
    plt.title("Embedding Distance vs Prediction Error")
    plt.savefig(save_path)
    plt.close()

# ======= kcat vs Km Scatter with Pearson =======
def plot_kcat_km_scatter(y_true, y_pred, save_path='scatter_kcat_km.png'):
    kcat_pred = y_pred[:, 0]
    km_pred = y_pred[:, 1]
    kcat_true = y_true[:, 0]
    km_true = y_true[:, 1]
    plt.figure(figsize=(8, 6))
    plt.scatter(kcat_pred, km_pred, label='Predicted', alpha=0.6)
    plt.scatter(kcat_true, km_true, label='True', alpha=0.6)
    plt.xlabel('kcat')
    plt.ylabel('Km')
    plt.legend()
    plt.title('kcat vs Km Prediction Distribution')
    plt.savefig(save_path)
    plt.close()

    r = np.corrcoef(kcat_pred, km_pred)[0,1]
    with open(save_path.replace('.png', '_pearson.txt'), 'w') as f:
        f.write(f"Pearson correlation between kcat and Km predictions: {r:.4f}\n")

# ======= t-SNE Visualization (with corrected colorbar) =======
def tsne_visualization(model, data_loader, device, save_path='tsne_kcat.png', param_index=0, max_points=1000):
    model.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            emb = model.get_graph_embedding(batch)
            embeddings.append(emb.cpu())
            labels.append(batch.y.view(-1, 2)[:, param_index].cpu())
    X = torch.cat(embeddings).numpy()
    Y = torch.cat(labels).numpy()
    X, Y = shuffle(X, Y, random_state=42)
    X, Y = X[:max_points], Y[:max_points]

    tsne = TSNE(n_components=2, perplexity=30)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(10, 8))
    sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=Y, cmap='coolwarm', s=60)
    plt.colorbar(sc)
    plt.title("Graph Embedding t-SNE (colored by target)")
    plt.savefig(save_path)
    plt.close()

# ======= 主入口 =======
if __name__ == '__main__':
    DATASET_PATH = '../data/processed/dataset_NAN_nopqr.pt'
    MODEL_PATH = '../outputs/nopqr/best_model.pt'
    SAVE_DIR = '../analysis_results/nopqr_more1'
    DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    BATCH_SIZE = 8

    dataset = torch.load(DATASET_PATH)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = PocketGNN1(node_input_dim=dataset[0].x.shape[1], edge_input_dim=dataset[0].edge_attr.shape[1])
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)

    # ==== 残差分析 & 多目标散点图 ====
    all_y_true, all_y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            y = batch.y.view(-1, 2).cpu()
            pred = model(batch).cpu()
            all_y_true.append(y)
            all_y_pred.append(pred)

    all_y_true = torch.cat(all_y_true).numpy()
    all_y_pred = torch.cat(all_y_pred).numpy()
    residual_analysis(all_y_true, all_y_pred, SAVE_DIR)
    plot_kcat_km_scatter(all_y_true, all_y_pred, os.path.join(SAVE_DIR, 'scatter_kcat_km.png'))

    # ==== t-SNE for kcat and Km ====
    tsne_visualization(model, loader, DEVICE, save_path=os.path.join(SAVE_DIR, 'tsne_kcat.png'), param_index=0)
    tsne_visualization(model, loader, DEVICE, save_path=os.path.join(SAVE_DIR, 'tsne_km.png'), param_index=1)

    # ==== 单个样本归因分析 ====
    data = dataset[0].to(DEVICE)
    attribution = compute_input_gradient(model, data, target_index=0)
    visualize_node_attribution(attribution, save_path=os.path.join(SAVE_DIR, 'node_importance_bar.png'))
    residue_class_importance(attribution, data, save_path=os.path.join(SAVE_DIR, 'residue_class_bar.png'))

    # ==== 平均残基贡献分析 ====
    average_residue_contribution(dataset, model, DEVICE, target_index=0, save_path=os.path.join(SAVE_DIR, 'residue_class_avg.png'))

    # ==== 热图归因分析（样本 × 残基）====
    attribution_matrix_heatmap(dataset, model, DEVICE, target_index=0, save_path=os.path.join(SAVE_DIR, 'residue_heatmap.png'))

    # ==== 嵌入距离 vs 预测误差分析 ====
    embedding_distance_vs_error(dataset, model, DEVICE, save_path=os.path.join(SAVE_DIR, 'embedding_error_scatter.png'))


/tmp/ipykernel_3467651/3899303956.py:182: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset = torch.load(DATASET_PATH)
/tmp/ipykernel_3467651/3899303956.py:186: FutureWa